# Inspect Inner Monologue from Trained SoRL Wrapper\n\nLoad a trained checkpoint, run batched generation with `K≠None` on GSM8K test set,\nand inspect the interleaved abstract (inner-monologue) tokens alongside NL output.

In [ ]:
import os
import torch
from transformers import AutoTokenizer
from safetensors.torch import load_file as load_safetensors

from sorl.sorl_wrapper import SorlModelWrapper, left_pad_and_mask
from data.pt_dataset import get_dataset

## 1. Configuration

In [ ]:
# ---- Edit these ----
CKPT_DIR = "./ckpt/ablate_XXXXXXXX_XXXX/exp8_1gpu_bs8_info1.0_abs0.5/final"  # <-- path to 'final' folder
MODEL_NAME = "Qwen/Qwen3-1.7B"
ABSTRACT_VOCAB_SIZE = 128
K = 4                    # abstract token every K trajectory tokens
MAX_NEW_TOKENS = 256
EVAL_BATCH_SIZE = 8
NUM_SAMPLES = 20         # how many GSM8K test samples to inspect
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 2. Load Trained Model

In [ ]:
def load_checkpoint(model_name, abstract_vocab_size, ckpt_dir, device):
    """Load SorlModelWrapper + checkpoint weights."""
    print(f"Loading base model: {model_name}")
    wrapper = SorlModelWrapper.from_pretrained(
        model_name,
        abstract_vocab_size_list=[abstract_vocab_size],
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_vocab = wrapper.vocab_sizes[0].item()

    # 1. Load full model weights from model.safetensors
    safetensors_path = os.path.join(ckpt_dir, "model.safetensors")
    if os.path.exists(safetensors_path):
        print(f"Loading weights from: {safetensors_path}")
        state = load_safetensors(safetensors_path, device="cpu")
        missing, unexpected = wrapper.load_state_dict(state, strict=False)
        print(f"  Loaded {len(state)} tensors (missing={len(missing)}, unexpected={len(unexpected)})")
    else:
        print(f"No model.safetensors found in {ckpt_dir}")

    # 2. Load abstract embedding rows from abs_embeddings.pt
    abs_path = os.path.join(ckpt_dir, "abs_embeddings.pt")
    if os.path.exists(abs_path):
        print(f"Loading abstract embeddings from: {abs_path}")
        ckpt = torch.load(abs_path, map_location="cpu", weights_only=False)
        hf = wrapper.model
        embed_w = hf.model.embed_tokens.weight if hasattr(hf, "model") else hf.transformer.wte.weight
        lm_head_w = hf.lm_head.weight
        embed_w.data[base_vocab:] = ckpt["embed_tokens"]
        lm_head_w.data[base_vocab:] = ckpt["lm_head"]
        print(f"  Restored abstract rows: embed={ckpt['embed_tokens'].shape}, lm_head={ckpt['lm_head'].shape}")
        print(f"  Step: {ckpt.get('step', '?')}, Epoch: {ckpt.get('epoch', '?')}")

    # 3. Load LoRA adapter if present
    adapter_config = os.path.join(ckpt_dir, "adapter_config.json")
    if os.path.exists(adapter_config):
        print(f"Loading LoRA adapter from: {ckpt_dir}")
        from peft import PeftModel
        wrapper.model = PeftModel.from_pretrained(wrapper.model, ckpt_dir)

    wrapper = wrapper.to(device).eval()
    return wrapper, tokenizer, base_vocab

wrapper, tokenizer, base_vocab = load_checkpoint(MODEL_NAME, ABSTRACT_VOCAB_SIZE, CKPT_DIR, DEVICE)
print(f"\nBase vocab: {base_vocab} | Total vocab: {wrapper.model.config.vocab_size}")
print(f"Abstract token IDs: [{base_vocab}, {wrapper.model.config.vocab_size})")

## 3. Load GSM8K Test Set

In [ ]:
dataset = get_dataset("gsm8k", split="train", tokenizer=tokenizer, max_length=512)
print(f"Dataset size: {len(dataset)}")
print(f"Using first {NUM_SAMPLES} samples for inspection")

## 4. Batched Generation with K≠None (Inner Monologue)

In [ ]:
@torch.no_grad()
def batched_generate(wrapper, tokenizer, dataset, num_samples, batch_size, K, max_new_tokens, device):
    """Generate with K≠None and return per-sample results with raw token ids."""
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    base_vocab = wrapper.vocab_sizes[0].item()
    extract_fn = getattr(dataset, "extract_answer", None)
    results = []

    for bs_start in range(0, num_samples, batch_size):
        bs_end = min(bs_start + batch_size, num_samples)
        prompts, prompt_lens, ref_texts = [], [], []

        for i in range(bs_start, bs_end):
            sample = dataset[i]
            pl = sample["prompt_len"]
            prompts.append(sample["input_ids"][:pl])
            prompt_lens.append(pl)
            ref_ids = sample["input_ids"][sample["input_ids"] < base_vocab]
            ref_texts.append(tokenizer.decode(ref_ids, skip_special_tokens=True))

        input_ids, attn_mask = left_pad_and_mask(prompts, pad_id=pad_id)
        input_ids, attn_mask = input_ids.to(device), attn_mask.to(device)

        generated = wrapper.generate(
            input_ids=input_ids,
            attention_mask=attn_mask,
            max_new_tokens=max_new_tokens,
            temperature=0.0, K=K, free_form=False,
        )

        max_pl = input_ids.size(1)
        for j, i in enumerate(range(bs_start, bs_end)):
            pad_len = max_pl - prompt_lens[j]
            gen_ids = generated[j, pad_len:]  # full sequence (prompt + generated)
            new_ids = generated[j, max_pl:]   # only newly generated tokens

            # Decode NL-only (strip abstract tokens)
            nl_ids = gen_ids[gen_ids < base_vocab]
            full_text = tokenizer.decode(nl_ids, skip_special_tokens=True)

            # Extract answer
            pred = extract_fn(full_text) if extract_fn else None
            gold = extract_fn(ref_texts[j]) if extract_fn else None
            hit = pred is not None and gold is not None and pred.strip() == gold.strip()

            results.append({
                "idx": i,
                "question": tokenizer.decode(prompts[j], skip_special_tokens=True),
                "ref_text": ref_texts[j],
                "gold": gold,
                "pred": pred,
                "correct": hit,
                "gen_ids": new_ids.cpu(),          # raw generated token ids (with abstract tokens)
                "full_text_nl_only": full_text,     # NL-only decoded text
            })

        print(f"  [{bs_end}/{num_samples}] done")

    correct = sum(r["correct"] for r in results)
    total = sum(1 for r in results if r["gold"] is not None)
    print(f"\nAccuracy (K={K}): {correct}/{total} = {correct/max(total,1)*100:.1f}%")
    return results

results = batched_generate(wrapper, tokenizer, dataset, NUM_SAMPLES, EVAL_BATCH_SIZE, K, MAX_NEW_TOKENS, DEVICE)

## 5. Inspect Inner Monologue\n\nFor each generated sequence, visualize the interleaving of NL tokens and abstract (inner-monologue) tokens.\nAbstract tokens are shown as `[ABS_<offset>]` with their relative ID within the abstract vocab.

In [ ]:
def format_token_stream(gen_ids, tokenizer, base_vocab):
    """Format generated token ids into a readable string with abstract tokens highlighted."""
    parts = []
    nl_buffer = []

    for tid in gen_ids.tolist():
        if tid < base_vocab:
            nl_buffer.append(tid)
        else:
            # Flush NL buffer
            if nl_buffer:
                parts.append(tokenizer.decode(nl_buffer, skip_special_tokens=True))
                nl_buffer = []
            abs_offset = tid - base_vocab
            parts.append(f"⟨ABS_{abs_offset}⟩")

    # Flush remaining NL tokens
    if nl_buffer:
        parts.append(tokenizer.decode(nl_buffer, skip_special_tokens=True))

    return "".join(parts)


def token_stats(gen_ids, base_vocab):
    """Count NL vs abstract tokens in generated sequence."""
    total = len(gen_ids)
    n_abs = (gen_ids >= base_vocab).sum().item()
    n_nl = total - n_abs
    return {"total": total, "nl": n_nl, "abstract": n_abs, "abs_ratio": n_abs / max(total, 1)}


for r in results:
    idx = r["idx"]
    stats = token_stats(r["gen_ids"], base_vocab)
    stream = format_token_stream(r["gen_ids"], tokenizer, base_vocab)

    print(f"{'='*80}")
    print(f"Sample {idx} | Correct: {r['correct']} | Gold: {r['gold']} | Pred: {r['pred']}")
    print(f"Tokens: {stats['total']} total, {stats['nl']} NL, {stats['abstract']} abstract ({stats['abs_ratio']:.1%})")
    print(f"{'-'*80}")
    print(f"Q: {r['question'][:200]}")
    print(f"{'-'*80}")
    print(f"Generated (with inner monologue):")
    print(stream[:1000])
    print()

## 6. Abstract Token Distribution\n\nWhich abstract token IDs are used most frequently? Are certain abstract tokens preferred?

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# Collect all abstract token offsets across all samples
abs_counter = Counter()
for r in results:
    for tid in r["gen_ids"].tolist():
        if tid >= base_vocab:
            abs_counter[tid - base_vocab] += 1

if abs_counter:
    offsets, counts = zip(*sorted(abs_counter.items()))
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(offsets, counts, color="steelblue", edgecolor="none")
    ax.set_xlabel("Abstract Token Offset (0-indexed)")
    ax.set_ylabel("Frequency")
    ax.set_title(f"Abstract Token Usage Distribution (K={K}, {NUM_SAMPLES} samples)")
    plt.tight_layout()
    plt.show()

    print(f"Unique abstract tokens used: {len(abs_counter)} / {ABSTRACT_VOCAB_SIZE}")
    print(f"Top 10: {abs_counter.most_common(10)}")
else:
    print("No abstract tokens generated — check that K is set and model is trained.")

## 7. Compare: K=None (NL-only) vs K≠None (with inner monologue)\n\nRun the same samples without abstract tokens to see if inner monologue changes accuracy.

In [ ]:
results_nl = batched_generate(wrapper, tokenizer, dataset, NUM_SAMPLES, EVAL_BATCH_SIZE, K=None, max_new_tokens=MAX_NEW_TOKENS, device=DEVICE)

# Side-by-side comparison
print(f"\n{'='*80}")
print(f"{'Sample':<8} {'K=None':<12} {'K='+str(K):<12} {'Gold':<12}")
print(f"{'='*80}")
for r_nl, r_k in zip(results_nl, results):
    mark_nl = "✓" if r_nl["correct"] else "✗"
    mark_k  = "✓" if r_k["correct"]  else "✗"
    print(f"{r_nl['idx']:<8} {mark_nl + ' ' + str(r_nl['pred']):<12} {mark_k + ' ' + str(r_k['pred']):<12} {r_k['gold']:<12}")

correct_nl = sum(r["correct"] for r in results_nl)
correct_k  = sum(r["correct"] for r in results)
total = sum(1 for r in results if r["gold"] is not None)
print(f"\nK=None: {correct_nl}/{total} = {correct_nl/max(total,1)*100:.1f}%")
print(f"K={K}:   {correct_k}/{total} = {correct_k/max(total,1)*100:.1f}%")

## 8. Inspect Specific Samples Where K≠None Differs from K=None

In [ ]:
# Show samples where K≠None and K=None disagree
diffs = [(r_nl, r_k) for r_nl, r_k in zip(results_nl, results) if r_nl["correct"] != r_k["correct"]]
print(f"Disagreements: {len(diffs)} / {len(results)}\n")

for r_nl, r_k in diffs:
    idx = r_nl["idx"]
    stats = token_stats(r_k["gen_ids"], base_vocab)
    stream = format_token_stream(r_k["gen_ids"], tokenizer, base_vocab)

    print(f"{'='*80}")
    print(f"Sample {idx} | Gold: {r_k['gold']}")
    print(f"  K=None → pred={r_nl['pred']}  {'✓' if r_nl['correct'] else '✗'}")
    print(f"  K={K}   → pred={r_k['pred']}  {'✓' if r_k['correct'] else '✗'}")
    print(f"  Abstract tokens: {stats['abstract']} ({stats['abs_ratio']:.1%})")
    print(f"{'-'*80}")
    print(f"Q: {r_k['question'][:200]}")
    print(f"\nK=None response:")
    print(f"  {r_nl['full_text_nl_only'][len(r_nl['question']):].strip()[:400]}")
    print(f"\nK={K} response (with inner monologue):")
    print(f"  {stream[:600]}")
    print()

# Inspect Inner Monologue from Trained SoRL Wrapper\n\nLoad a trained checkpoint, run batched generation with `K≠None` on GSM8K test set,\nand inspect the interleaved abstract (inner-monologue) tokens alongside NL output.